# Gemma 4 H/k v2_planact Sweep Notebook

Dieses Notebook startet den H/k-Agent-Runner fuer den 12-Task-Sweep mit `gemma4:26b` als Planner und `gemma4:e4b` als Executor. Standardmaessig laeuft es ueber den lokalen Ollama-kompatiblen Proxy zur Google Vertex AI MaaS API.

Methodischer Hinweis fuer die Masterarbeit: `v2_planact` ist eine Architekturvariante zur besseren Executor-Grounding-/Replanning-Kontrolle. Die WebArena-Verified Task-Prompts, Task-Contracts und finale offizielle Evaluation bleiben die Grundlage. Die zusaetzlichen PlanAct-Regeln duerfen keine Gold-Antworten, keine Evaluator-Metadaten und keine task-spezifisch hardcodierten Loesungen einfuehren.

Der Runner darf weiterhin `gemma4:26b` und `gemma4:e4b` als Modellnamen loggen; der Proxy routet beide Namen intern auf `VERTEX_MAAS_MODEL`, z.B. `gemma-4-26b-a4b-it-maas`. Wenn du stattdessen lokal mit Ollama laufen willst, setze `USE_VERTEX_PROXY = False`.


In [37]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

dataset_path = ROOT / "external/webarena-verified/assets/dataset/webarena-verified.json"
hard_subset_path = ROOT / "external/webarena-verified/assets/dataset/subsets/webarena-verified-hard.json"

supported_single_sites = {"gitlab", "reddit", "shopping", "shopping_admin"}

all_tasks = json.loads(dataset_path.read_text())
hard_ids = {int(x) for x in json.loads(hard_subset_path.read_text())["task_ids"]}

TASK_IDS = sorted(
    int(t["task_id"])
    for t in all_tasks
    if int(t["task_id"]) in hard_ids
    and len(t.get("sites", [])) == 1
    and t["sites"][0] in supported_single_sites
)

TASK_IDS


[11,
 15,
 21,
 25,
 28,
 29,
 31,
 42,
 44,
 50,
 63,
 64,
 65,
 66,
 67,
 68,
 96,
 105,
 106,
 108,
 110,
 111,
 113,
 116,
 124,
 125,
 127,
 142,
 143,
 147,
 148,
 149,
 156,
 157,
 163,
 165,
 166,
 170,
 171,
 172,
 184,
 185,
 186,
 191,
 193,
 196,
 197,
 200,
 204,
 212,
 214,
 226,
 235,
 240,
 259,
 269,
 271,
 273,
 284,
 286,
 293,
 296,
 297,
 301,
 303,
 304,
 307,
 316,
 320,
 321,
 323,
 325,
 327,
 328,
 335,
 337,
 338,
 343,
 345,
 349,
 350,
 357,
 375,
 387,
 388,
 397,
 398,
 399,
 400,
 401,
 407,
 408,
 409,
 410,
 415,
 416,
 417,
 418,
 431,
 432,
 435,
 441,
 443,
 444,
 446,
 449,
 451,
 466,
 488,
 489,
 491,
 493,
 499,
 502,
 505,
 507,
 508,
 509,
 519,
 520,
 521,
 522,
 523,
 525,
 526,
 528,
 529,
 530,
 544,
 545,
 546,
 549,
 550,
 551,
 571,
 572,
 576,
 577,
 578,
 580,
 581,
 584,
 585,
 586,
 587,
 590,
 591,
 592,
 596,
 597,
 598,
 603,
 610,
 615,
 617,
 618,
 624,
 625,
 634,
 636,
 638,
 640,
 644,
 645,
 646,
 647,
 651,
 658,
 659,
 66

In [38]:
from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys
import time
from urllib.request import Request, urlopen

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Targeted v3_repair_llm GitLab-MUTATE run. This keeps it separate from older v3/v3_repair_brief artifacts.
EXPERIMENT_NAME = 'hk-agent-browsergym-planact-main-v02_basis_v3'
#TASK_IDS = [44, 157, 105, 522, 800, 444, 407, 644, 28, 387, 795, 507, 505, 15, 108]
#TASK_IDS = [44, 157, 105, 522, 800, 444, 407, 644, 28, 387]  # smaller sweep for debugging
HS = [0, 2, 5, 10]
KS = [0, 2, 5, 10]
RUN_MODE = 'agent'
# Wenn True, druckt die Run-Zelle nur den fertigen Terminal-Befehl und startet keinen Sweep.
PRINT_COMMAND_ONLY = True
# Wenn True, bewahrt der Runner alte summary-Zeilen und ueberspringt vorhandene Run-Artefakte.
RESUME_SUMMARY = True
# Sauber, aber langsam: startet bei offiziellen MUTATE-Tasks den jeweiligen WebArena-Service neu.
RESET_SITE_BEFORE_MUTATE = False
SITE_RESET_TIMEOUT_SECONDS = 180
# Transparente Korrektur: eindeutige Suffix-/URL-Normalisierungen zaehlen als korrigierter official_success; raw Wert bleibt als official_success_raw erhalten.
USE_CONTAMINATION_ADJUSTED_SUCCESS = True
# In dieser Notebook-Auswertung werden transparente Evaluator-Normalisierungen direkt als official_success gezaehlt.
PROMOTE_ADJUSTED_TO_OFFICIAL_IN_NOTEBOOK = True
SUCCESS_POLICY = 'contamination_adjusted' if USE_CONTAMINATION_ADJUSTED_SUCCESS else 'webarena'


def apply_analysis_success_columns(frame, *, use_adjusted=None, promote_adjusted_to_official=None):
    """Add notebook-level success columns and apply transparent evaluator corrections.

    If the stored success values are stale, this recomputes transparent
    adjusted evaluator cases from run artifacts. The
    corrected value is written to `official_success` for this notebook
    analysis. The original CSV values are kept as `official_success_raw` and
    `evaluation_success_raw`.
    """
    import pandas as pd

    use_adjusted = USE_CONTAMINATION_ADJUSTED_SUCCESS if use_adjusted is None else use_adjusted
    promote_adjusted_to_official = (
        PROMOTE_ADJUSTED_TO_OFFICIAL_IN_NOTEBOOK
        if promote_adjusted_to_official is None
        else promote_adjusted_to_official
    )

    def _as_bool(series):
        return series.fillna(False).astype(str).str.lower().isin({'true', '1', 'yes', 'success'})

    if 'official_success' in frame.columns and 'official_success_raw' not in frame.columns:
        frame['official_success_raw'] = frame['official_success']
    if 'official_success_raw' in frame.columns:
        frame['official_success_raw_bool'] = _as_bool(frame['official_success_raw'])
    elif 'official_success' in frame.columns:
        frame['official_success_raw_bool'] = _as_bool(frame['official_success'])
    else:
        frame['official_success_raw_bool'] = pd.Series(False, index=frame.index)
    frame['official_success_bool'] = frame['official_success_raw_bool']

    if 'evaluation_success' in frame.columns and 'evaluation_success_raw' not in frame.columns:
        frame['evaluation_success_raw'] = frame['evaluation_success']

    if 'contamination_adjusted_success' in frame.columns:
        frame['contamination_adjusted_success_bool'] = frame['official_success_bool'] | _as_bool(frame['contamination_adjusted_success'])
    else:
        frame['contamination_adjusted_success_bool'] = frame['official_success_bool']

    if use_adjusted and 'output_dir' in frame.columns:
        if str(ROOT / 'scripts') not in sys.path:
            sys.path.insert(0, str(ROOT / 'scripts'))
        try:
            from hk_agent.diagnostics import contamination_adjusted_eval_diagnostics
        except Exception as exc:
            print('contamination_adjusted_eval_diagnostics konnte nicht importiert werden:', exc)
        else:
            if 'failure_category' in frame.columns:
                category = frame['failure_category'].fillna('').astype(str)
                recompute_candidate = category.isin({'official_eval_mismatch', 'contamination_suffix_near_miss'})
            else:
                recompute_candidate = pd.Series(False, index=frame.index)
            if 'final_response_status' in frame.columns:
                recompute_candidate = recompute_candidate | frame['final_response_status'].fillna('').astype(str).eq('SUCCESS')
            needs_recompute = ~frame['official_success_bool'] & ~frame['contamination_adjusted_success_bool'] & recompute_candidate
            for idx in frame.index[needs_recompute]:
                output_dir = frame.at[idx, 'output_dir']
                if pd.isna(output_dir) or not str(output_dir):
                    continue
                diag = contamination_adjusted_eval_diagnostics(Path(str(output_dir)))
                if bool(diag.get('contamination_adjusted_success')):
                    frame.at[idx, 'contamination_adjusted_success_bool'] = True
                    frame.at[idx, 'contamination_adjusted_success'] = True
                    for key, value in diag.items():
                        if key not in frame.columns:
                            frame[key] = pd.NA
                        frame.at[idx, key] = value

    if 'evaluation_success_raw' in frame.columns:
        evaluation_success_bool = _as_bool(frame['evaluation_success_raw'])
    elif 'evaluation_success' in frame.columns:
        evaluation_success_bool = _as_bool(frame['evaluation_success'])
    else:
        evaluation_success_bool = frame['official_success_bool']

    if use_adjusted:
        evaluation_success_bool = evaluation_success_bool | frame['contamination_adjusted_success_bool']

    frame['analysis_success_bool'] = evaluation_success_bool
    frame['evaluation_success'] = evaluation_success_bool
    if promote_adjusted_to_official:
        frame['official_success'] = evaluation_success_bool
        frame['official_success_bool'] = evaluation_success_bool
    frame['evaluation_success_policy_analysis'] = 'contamination_adjusted' if use_adjusted else 'webarena'
    return frame

# Aktualisiert alte summary-Zeilen aus vorhandenen Artefakten, ohne Aufgaben neu zu starten.
REFRESH_EXISTING_DIAGNOSTICS = True
# Wenn True, werden nur vorhandene Artefakte neu bewertet; fehlende H/K-Kombinationen werden nicht gestartet.
REFRESH_EXISTING_ONLY = False
PLANNER_MODEL = 'gemma4:26b'
EXECUTOR_MODEL = 'gemma4:e4b'
AGENT_ARCHITECTURE = 'v3'
MAX_STEPS_POLICY = 'tiered'
MAX_STEPS = 500  # fallback/fixed-budget value; tiered runs use the per-tier values below.
MAX_STEPS_BY_CAPABILITY_TIER = {
    'navigation': 20,
    'visible_retrieve': 30,
    'structured_retrieve': 30,
    'policy': 25,
    'mutation': 50,
}
PLANNER_CALL_MARGIN = 2
MAX_PLANNER_CALLS = 'auto'

def planner_call_budget_for_h(h, max_steps, margin=PLANNER_CALL_MARGIN):
    if max_steps <= 0:
        return None
    if h <= 0:
        return max(3, margin + 1)
    return math.ceil(max_steps / h) + margin

PLANNER_CALL_BUDGET_BY_TIER_AND_H = {
    tier: {h: planner_call_budget_for_h(h, max_steps) for h in HS}
    for tier, max_steps in MAX_STEPS_BY_CAPABILITY_TIER.items()
}
EXPECTED_RUNS = len(TASK_IDS) * len(HS) * len(KS)
LLM_TIMEOUT_SECONDS = 600
# 0 keeps full H/k execution comparable to older runs. Use 2 for quick smoke/debug runs.
MAX_CONSECUTIVE_LLM_TIMEOUTS = 0

# Prompt provenance: WebArena-Verified prompt basis + local architecture extension.
PLANNER_PROMPT_PATH = ROOT / 'prompts/planner_system.md'
PLANNER_USER_TEMPLATE_PATH = ROOT / 'prompts/prompt_user_template.md'
EXECUTOR_PROMPT_PATH = ROOT / 'prompts/executor_system.md'
WEBARENA_VERIFIED_PROMPT_ROOT = ROOT / 'external/webarena-verified/examples/prompts'
ARCHITECTURE_EXTENSION_PROMPT_ROOT = ROOT / 'prompts/v2'

# Default: run the same sweep through Google Vertex AI MaaS via the local proxy.
USE_VERTEX_PROXY = True
LOCAL_OLLAMA_URL = 'http://localhost:11434'
PROXY_URL = 'http://127.0.0.1:11435'

# Optional Vertex MaaS proxy settings, only used when USE_VERTEX_PROXY=True.
PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT', 'project-e784413d-cd26-4398-adf')
LOCATION = os.environ.get('GOOGLE_CLOUD_LOCATION', 'global')
VERTEX_MAAS_MODEL = os.environ.get('VERTEX_MAAS_MODEL', 'gemma-4-26b-a4b-it-maas')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION
os.environ['VERTEX_MAAS_MODEL'] = VERTEX_MAAS_MODEL
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

ROOT, EXPERIMENT_NAME, AGENT_ARCHITECTURE, TASK_IDS, HS, KS, MAX_STEPS_POLICY, MAX_STEPS_BY_CAPABILITY_TIER, MAX_PLANNER_CALLS, PLANNER_CALL_BUDGET_BY_TIER_AND_H, EXPECTED_RUNS, PLANNER_MODEL, EXECUTOR_MODEL, USE_VERTEX_PROXY, PRINT_COMMAND_ONLY, RESUME_SUMMARY


(PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code'),
 'hk-agent-browsergym-planact-main-v02_basis_v3',
 'v3',
 [11,
  15,
  21,
  25,
  28,
  29,
  31,
  42,
  44,
  50,
  63,
  64,
  65,
  66,
  67,
  68,
  96,
  105,
  106,
  108,
  110,
  111,
  113,
  116,
  124,
  125,
  127,
  142,
  143,
  147,
  148,
  149,
  156,
  157,
  163,
  165,
  166,
  170,
  171,
  172,
  184,
  185,
  186,
  191,
  193,
  196,
  197,
  200,
  204,
  212,
  214,
  226,
  235,
  240,
  259,
  269,
  271,
  273,
  284,
  286,
  293,
  296,
  297,
  301,
  303,
  304,
  307,
  316,
  320,
  321,
  323,
  325,
  327,
  328,
  335,
  337,
  338,
  343,
  345,
  349,
  350,
  357,
  375,
  387,
  388,
  397,
  398,
  399,
  400,
  401,
  407,
  408,
  409,
  410,
  415,
  416,
  417,
  418,
  431,
  432,
  435,
  441,
  443,
  444,
  446,
  449,
  451,
  466,
  488,
  489,
  491,
  493,
  499,
  502,
  505,
  507,
  508,
  509,
  519,
  520,
  521,
  522,
  523,
  525,
  526,
  5

## 0. WebArena-Docker-Services automatisch neu starten

Diese Zelle startet die WebArena-Services fuer die aktuelle Experimentliste neu. Das ist besonders vor MUTATE-Retests hilfreich, weil GitLab/Shopping/Reddit sonst bereits veraenderte Daten enthalten koennen. Wenn du nur den Befehl sehen willst, setze `AUTO_RESTART_WEBARENA_CONTAINERS = False`.


In [42]:
# Achtung: `env start` entfernt den bestehenden Container des jeweiligen Sites und startet ihn frisch.
# Wenn diese Zelle laeuft, werden die Services automatisch neu gestartet.
AUTO_RESTART_WEBARENA_CONTAINERS = True
RESTART_WITH_DIRECT_DOCKER_RUN = True  # True = docker rm/run statt webarena-verified env start.
RESTART_NEEDED_SITES_EVEN_IF_STOPPED = True  # True = fuer TASK_IDS benoetigte Sites ebenfalls starten.
AUTO_CLEAR_WEBARENA_PORT_CONFLICTS = True  # Stoppt Docker-Container, die WebArena-Ports blockieren.
DOCKER_PLATFORM = 'linux/amd64'  # Apple Silicon: verhindert Platform-Warnungen bei WebArena-Images.
WEBARENA_SITES_TO_CHECK = ['shopping', 'shopping_admin', 'reddit', 'gitlab']
WEBARENA_SITE_HOST_PORTS = {
    'shopping': [7770, 7771],
    'shopping_admin': [7780, 7781],
    'reddit': [9999, 9998],
    'gitlab': [8023, 8024],
}
WEBARENA_DOCKER_CONFIG = {
    'shopping': {
        'name': 'webarena-verified-shopping',
        'cli_name': 'webarena_verified_shopping',
        'ports': ['7770:80', '7771:8877'],
        'image': 'am1n3e/webarena-verified-shopping',
    },
    'shopping_admin': {
        'name': 'webarena-verified-shopping_admin',
        'cli_name': 'webarena_verified_shopping_admin',
        'ports': ['7780:80', '7781:8877'],
        'image': 'am1n3e/webarena-verified-shopping_admin',
    },
    'reddit': {
        'name': 'webarena-verified-reddit',
        'cli_name': 'webarena_verified_reddit',
        'ports': ['9999:80', '9998:8877'],
        'image': 'am1n3e/webarena-verified-reddit',
    },
    'gitlab': {
        'name': 'webarena-verified-gitlab',
        'cli_name': 'webarena_verified_gitlab',
        'ports': ['8023:8023', '8024:8877'],
        'image': 'am1n3e/webarena-verified-gitlab',
    },
}

wa_root = ROOT / 'external/webarena-verified'
assert wa_root.exists(), wa_root

def docker_containers_on_ports(ports):
    proc = subprocess.run(
        ['docker', 'ps', '--format', '{{.ID}}\t{{.Names}}\t{{.Ports}}'],
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        return []
    markers = [f':{port}->' for port in ports]
    conflicts = []
    for line in proc.stdout.splitlines():
        parts = line.split('\t', 2)
        if len(parts) != 3:
            continue
        container_id, name, published_ports = parts
        if any(marker in published_ports for marker in markers):
            conflicts.append({'id': container_id, 'name': name, 'ports': published_ports})
    return conflicts

def clear_port_conflicts_for_site(site):
    conflicts = docker_containers_on_ports(WEBARENA_SITE_HOST_PORTS.get(site, []))
    if not conflicts:
        return []
    print(f'Port conflicts for {site}:', conflicts)
    if not AUTO_CLEAR_WEBARENA_PORT_CONFLICTS:
        return conflicts
    for conflict in conflicts:
        print('Stopping/removing conflicting container:', conflict)
        subprocess.run(['docker', 'stop', conflict['id']], text=True, capture_output=True, check=False)
        subprocess.run(['docker', 'rm', '-f', conflict['id']], text=True, capture_output=True, check=False)
    return conflicts

def remove_known_webarena_containers(site):
    config = WEBARENA_DOCKER_CONFIG[site]
    for name in [config['name'], config['cli_name']]:
        subprocess.run(['docker', 'rm', '-f', name], text=True, capture_output=True, check=False)

def direct_docker_run_command(site):
    config = WEBARENA_DOCKER_CONFIG[site]
    cmd = ['docker', 'run', '-d', '--platform', DOCKER_PLATFORM, '--name', config['name']]
    for port in config['ports']:
        cmd.extend(['-p', port])
    cmd.append(config['image'])
    return cmd

running_sites = []
for site in WEBARENA_SITES_TO_CHECK:
    status_cmd = ['uv', 'run', 'webarena-verified', 'env', 'status', '--site', site]
    status_proc = subprocess.run(status_cmd, cwd=wa_root, text=True, capture_output=True, check=False)
    if status_proc.returncode == 0:
        running_sites.append(site)

sites_needed = []
if RESTART_NEEDED_SITES_EVEN_IF_STOPPED:
    if str(ROOT / 'scripts') not in sys.path:
        sys.path.insert(0, str(ROOT / 'scripts'))
    from hk_agent.task_loader import select_tasks

    selected_tasks = select_tasks(
        ROOT / 'external/webarena-verified',
        TASK_IDS,
        single_site_only=True,
        supported_sites_only=True,
        allow_non_hard_task_ids=False,
    )
    for task in selected_tasks:
        sites_needed.extend(task.sites or [task.primary_site])
sites_to_restart = sorted(set(running_sites) | {site for site in sites_needed if site})
if not sites_to_restart:
    sites_to_restart = WEBARENA_SITES_TO_CHECK

restart_plan = []
for site in sites_to_restart:
    if RESTART_WITH_DIRECT_DOCKER_RUN:
        start_cmd = direct_docker_run_command(site)
    else:
        start_cmd = [
            'uv', 'run', 'webarena-verified', 'env', 'start',
            '--site', site,
            '--timeout', str(SITE_RESET_TIMEOUT_SECONDS),
        ]
    restart_plan.append({'site': site, 'cmd': start_cmd})

print('running_sites:', running_sites)
print('sites_needed_by_TASK_IDS:', sorted({site for site in sites_needed if site}))
print('WebArena restart plan:')
for item in restart_plan:
    print(f"- {item['site']}:", ' '.join(item['cmd']))

if AUTO_RESTART_WEBARENA_CONTAINERS:
    for item in restart_plan:
        print(f"\nRestarting {item['site']} ...")
        if RESTART_WITH_DIRECT_DOCKER_RUN:
            remove_known_webarena_containers(item['site'])
            clear_port_conflicts_for_site(item['site'])
            proc = subprocess.run(item['cmd'], cwd=ROOT, text=True, capture_output=True, check=False)
        else:
            stop_cmd = ['uv', 'run', 'webarena-verified', 'env', 'stop', '--site', item['site']]
            subprocess.run(stop_cmd, cwd=wa_root, text=True, capture_output=True, check=False)
            proc = subprocess.run(item['cmd'], cwd=wa_root, text=True, capture_output=True, check=False)
        print(proc.stdout[-2000:])
        if proc.returncode != 0:
            err = (proc.stderr or proc.stdout or '')
            print(err[-4000:])
            if 'port is already allocated' in err.lower() or 'bind for' in err.lower():
                clear_port_conflicts_for_site(item['site'])
                print(f"Retrying {item['site']} after clearing port conflicts ...")
                proc = subprocess.run(item['cmd'], cwd=ROOT if RESTART_WITH_DIRECT_DOCKER_RUN else wa_root, text=True, capture_output=True, check=False)
                print(proc.stdout[-2000:])
                if proc.returncode != 0:
                    print((proc.stderr or proc.stdout or '')[-4000:])
                    raise RuntimeError(f"Restart retry failed for {item['site']} with return code {proc.returncode}")
            else:
                raise RuntimeError(f"Restart failed for {item['site']} with return code {proc.returncode}")
    print('\nWebArena services restarted:', ', '.join(item['site'] for item in restart_plan))
else:
    print('\nAUTO_RESTART_WEBARENA_CONTAINERS=False; es wurde nichts neu gestartet.')


running_sites: []
sites_needed_by_TASK_IDS: ['gitlab', 'reddit', 'shopping', 'shopping_admin']
WebArena restart plan:
- gitlab: docker run -d --platform linux/amd64 --name webarena-verified-gitlab -p 8023:8023 -p 8024:8877 am1n3e/webarena-verified-gitlab
- reddit: docker run -d --platform linux/amd64 --name webarena-verified-reddit -p 9999:80 -p 9998:8877 am1n3e/webarena-verified-reddit
- shopping: docker run -d --platform linux/amd64 --name webarena-verified-shopping -p 7770:80 -p 7771:8877 am1n3e/webarena-verified-shopping
- shopping_admin: docker run -d --platform linux/amd64 --name webarena-verified-shopping_admin -p 7780:80 -p 7781:8877 am1n3e/webarena-verified-shopping_admin

Restarting gitlab ...
d9c8b025b2000e46d99386e221879ec4d9359c0f6b67292937743ab416a3a58d


Restarting reddit ...
5eec66dcc82751227228eb58c35e29ded32558f1c55e9a52bfa945e091c0c00f


Restarting shopping ...
3a0a1295954df573eae2eecf8813da895418cc62fca59b76481b5ec3046ec368


Restarting shopping_admin ...
e63ae2f334

## 0b. MUTATE-Retest auf sauberem Zustand

Diese Zelle baut und optional startet kleine MUTATE-Retests. Standard ist `h2/k2`, weil diese Kombination aktuell die sinnvollste PlanAct-Variante testet. Wenn du zusaetzlich Baseline willst, setze `MUTATE_RETEST_HK_PAIRS = [(0, 0), (2, 2)]`.


In [4]:
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from hk_agent.task_loader import select_tasks
from hk_agent.capabilities import infer_official_task_type

RUN_MUTATE_RETEST = False  # True = Retest direkt starten; False = nur Befehle ausgeben.
MUTATE_RETEST_EXPERIMENT_NAME = f'{EXPERIMENT_NAME}-mutate-retest-clean'
MUTATE_RETEST_HK_PAIRS = [(2, 2)]  # z.B. [(0, 0), (2, 2)] fuer Baseline plus PlanAct.
MUTATE_RETEST_RESET_BEFORE_MUTATE = False  # False, wenn die Restart-Zelle direkt davor gelaufen ist.

tasks_for_current_ids = select_tasks(
    ROOT / 'external/webarena-verified',
    TASK_IDS,
    single_site_only=True,
    supported_sites_only=True,
    allow_non_hard_task_ids=False,
)
MUTATE_RETEST_TASK_IDS = [
    int(task.task_id)
    for task in tasks_for_current_ids
    if infer_official_task_type(task.raw_task) == 'MUTATE'
]

print('mutate_retest_task_ids:', MUTATE_RETEST_TASK_IDS)
print('mutate_retest_hk_pairs:', MUTATE_RETEST_HK_PAIRS)

mutate_retest_commands = []
for h, k in MUTATE_RETEST_HK_PAIRS:
    retest_cmd = [
        'uv', 'run', 'python', 'scripts/run_hk_agent_experiment.py',
        '--experiment-name', MUTATE_RETEST_EXPERIMENT_NAME,
        '--task-ids', *[str(task_id) for task_id in MUTATE_RETEST_TASK_IDS],
        '--hs', str(h),
        '--ks', str(k),
        '--run-mode', RUN_MODE,
        '--planner-model', PLANNER_MODEL,
        '--executor-model', EXECUTOR_MODEL,
        '--agent-architecture', AGENT_ARCHITECTURE,
        '--max-steps-policy', MAX_STEPS_POLICY,
        '--max-steps-navigation', str(MAX_STEPS_BY_CAPABILITY_TIER['navigation']),
        '--max-steps-retrieval', str(MAX_STEPS_BY_CAPABILITY_TIER['structured_retrieve']),
        '--max-steps-policy-task', str(MAX_STEPS_BY_CAPABILITY_TIER['policy']),
        '--max-steps-mutation', str(MAX_STEPS_BY_CAPABILITY_TIER['mutation']),
        '--max-planner-calls', str(MAX_PLANNER_CALLS),
        '--planner-call-margin', str(PLANNER_CALL_MARGIN),
        '--max-steps', str(MAX_STEPS),
        '--llm-timeout-seconds', str(LLM_TIMEOUT_SECONDS),
        '--max-consecutive-llm-timeouts', str(MAX_CONSECUTIVE_LLM_TIMEOUTS),
        '--success-policy', SUCCESS_POLICY,
        '--resume-summary',
        '--refresh-existing-diagnostics',
    ]
    if USE_VERTEX_PROXY:
        retest_cmd.extend(['--ollama-base-url', PROXY_URL])
    else:
        retest_cmd.extend(['--ollama-base-url', LOCAL_OLLAMA_URL])
    if MUTATE_RETEST_RESET_BEFORE_MUTATE:
        retest_cmd.extend(['--reset-site-before-mutate', '--site-reset-timeout-seconds', str(SITE_RESET_TIMEOUT_SECONDS)])
    mutate_retest_commands.append(retest_cmd)

for retest_cmd in mutate_retest_commands:
    print('\n' + ' '.join(retest_cmd))

if RUN_MUTATE_RETEST:
    for retest_cmd in mutate_retest_commands:
        print('\nRunning:', ' '.join(retest_cmd))
        proc = subprocess.run(retest_cmd, cwd=ROOT, env=runner_env if 'runner_env' in globals() else None, text=True, check=False)
        if proc.returncode != 0:
            raise RuntimeError(f'MUTATE retest failed with return code {proc.returncode}')
    print('\nMUTATE retest summary:', ROOT / 'runs/hk-agent' / MUTATE_RETEST_EXPERIMENT_NAME / 'summary.csv')
else:
    print('\nRUN_MUTATE_RETEST=False; Befehle wurden nur gedruckt.')


mutate_retest_task_ids: [522, 800, 444, 407, 644, 795, 507, 505]
mutate_retest_hk_pairs: [(2, 2)]

uv run python scripts/run_hk_agent_experiment.py --experiment-name hk-agent-browsergym-planact-main-v02_basis_v3-mutate-retest-clean --task-ids 522 800 444 407 644 795 507 505 --hs 2 --ks 2 --run-mode agent --planner-model gemma4:26b --executor-model gemma4:e4b --agent-architecture v3 --max-steps-policy tiered --max-steps-navigation 45 --max-steps-retrieval 45 --max-steps-policy-task 45 --max-steps-mutation 150 --max-planner-calls auto --planner-call-margin 2 --max-steps 500 --llm-timeout-seconds 600 --max-consecutive-llm-timeouts 0 --success-policy contamination_adjusted --resume-summary --refresh-existing-diagnostics --ollama-base-url http://127.0.0.1:11435

RUN_MUTATE_RETEST=False; Befehle wurden nur gedruckt.


## 1. Prompt Provenance And Thesis Guardrails

Diese Zelle dokumentiert vor dem Lauf, welche Prompt-Dateien verwendet werden. Wichtig fuer die Nachvollziehbarkeit: Die Basis-Prompts kommen aus `external/webarena-verified/examples/prompts/`. `prompts/v2/` ergaenzt nur Architekturregeln fuer Grounding, Validierung, MUTATE-Sorgfalt und Artefakt-Logging.


In [5]:
import hashlib

assert AGENT_ARCHITECTURE == 'v3', AGENT_ARCHITECTURE
assert WEBARENA_VERIFIED_PROMPT_ROOT.exists(), WEBARENA_VERIFIED_PROMPT_ROOT
assert ARCHITECTURE_EXTENSION_PROMPT_ROOT.exists(), ARCHITECTURE_EXTENSION_PROMPT_ROOT

def sha256_short(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]

site_prompt_files = sorted({WEBARENA_VERIFIED_PROMPT_ROOT / f'{site}.md' for site in ['gitlab', 'reddit', 'shopping', 'shopping_admin']})
prompt_files = [
    PLANNER_PROMPT_PATH,
    PLANNER_USER_TEMPLATE_PATH,
    EXECUTOR_PROMPT_PATH,
    ARCHITECTURE_EXTENSION_PROMPT_ROOT / 'executor_base.md',
    *site_prompt_files,
]
records = []
for prompt_file in prompt_files:
    records.append({
        'path': str(prompt_file.relative_to(ROOT)),
        'exists': prompt_file.exists(),
        'sha256_16': sha256_short(prompt_file) if prompt_file.exists() else None,
        'role': 'webarena_verified_basis' if WEBARENA_VERIFIED_PROMPT_ROOT in prompt_file.parents else 'local_architecture_or_runner_prompt',
    })

print(json.dumps({
    'architecture': AGENT_ARCHITECTURE,
    'official_evaluator': 'WebArena-Verified eval-tasks',
    'prompt_policy': 'WebArena-Verified example prompts are rendered as the executor prompt basis; v2_planact appends grounding and validation rules only.',
    'no_gold_metadata_policy': 'Do not expose official eval answers or evaluator internals to the agent.',
    'prompt_files': records,
}, indent=2))

missing = [record['path'] for record in records if not record['exists']]
if missing:
    raise FileNotFoundError(missing)


{
  "architecture": "v3",
  "official_evaluator": "WebArena-Verified eval-tasks",
  "prompt_policy": "WebArena-Verified example prompts are rendered as the executor prompt basis; v2_planact appends grounding and validation rules only.",
  "no_gold_metadata_policy": "Do not expose official eval answers or evaluator internals to the agent.",
  "prompt_files": [
    {
      "path": "prompts/planner_system.md",
      "exists": true,
      "sha256_16": "114590e9856054b3",
      "role": "local_architecture_or_runner_prompt"
    },
    {
      "path": "prompts/prompt_user_template.md",
      "exists": true,
      "sha256_16": "9f7382993fcbcb5b",
      "role": "local_architecture_or_runner_prompt"
    },
    {
      "path": "prompts/executor_system.md",
      "exists": true,
      "sha256_16": "351a0bfeb3c2fcc5",
      "role": "local_architecture_or_runner_prompt"
    },
    {
      "path": "prompts/v2/executor_base.md",
      "exists": true,
      "sha256_16": "d7d84bdaa3bd2998",
      "role"

## 2. Optional Vertex Authentication Check

Diese Zelle ist nur notwendig, wenn `USE_VERTEX_PROXY = True` gesetzt ist. Fuer den Standardlauf mit lokalem Ollama kannst du sie ueberspringen.


In [6]:
if not USE_VERTEX_PROXY:
    print('USE_VERTEX_PROXY=False; ADC check skipped. Local Ollama will be used.')
else:
    import google.auth
    import google.auth.transport.requests
    from google.auth.exceptions import DefaultCredentialsError

    RUN_ADC_SETUP = False  # auf True setzen, wenn du den Google-ADC-Login aus dem Notebook starten willst
    RUN_GCLOUD_ADC_LOGIN = False  # Alternative, falls gcloud installiert ist
    ENABLE_VERTEX_AI_API = False  # optional: setzt gcloud voraus und braucht passende Rechte im Projekt

    print('PROJECT_ID:', PROJECT_ID)
    print('LOCATION:', LOCATION)
    print('VERTEX_MAAS_MODEL:', VERTEX_MAAS_MODEL)
    print('Login-E-Mail im Browser waehlen:', 'niclascramer@gmail.com')

    if RUN_ADC_SETUP:
        # Startet den offiziellen Google ADC Setup Flow. Das kann einen Browser/Login-Link oeffnen.
        subprocess.run(
            ['bash', '-lc', 'bash <(curl -sSL https://storage.googleapis.com/cloud-samples-data/adc/setup_adc.sh)'],
            check=True,
        )

    if RUN_GCLOUD_ADC_LOGIN:
        if not shutil.which('gcloud'):
            raise RuntimeError('gcloud ist nicht installiert. Nutze RUN_ADC_SETUP=True oder fuehre den curl-Befehl im Terminal aus.')
        subprocess.run(['gcloud', 'auth', 'application-default', 'login'], check=True)
        subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

    if ENABLE_VERTEX_AI_API:
        if not shutil.which('gcloud'):
            raise RuntimeError('gcloud ist nicht installiert; Vertex AI API bitte in der Cloud Console aktivieren.')
        subprocess.run(['gcloud', 'services', 'enable', 'aiplatform.googleapis.com', '--project', PROJECT_ID], check=True)

    try:
        credentials, detected_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
        credentials.refresh(google.auth.transport.requests.Request())
        print('ADC OK')
        print('configured_project:', PROJECT_ID)
        print('detected_project:', detected_project)
        print('token_prefix:', str(credentials.token)[:12] + '...')
    except DefaultCredentialsError as exc:
        print('ADC fehlt noch.')
        print('Empfohlen: Im Terminal ausfuehren, Login mit niclascramer@gmail.com waehlen, danach Kernel neu starten:')
        print('bash <(curl -sSL https://storage.googleapis.com/cloud-samples-data/adc/setup_adc.sh)')
        print('Oder in dieser Zelle RUN_ADC_SETUP = True setzen und erneut ausfuehren.')
        raise


PROJECT_ID: project-e784413d-cd26-4398-adf
LOCATION: global
VERTEX_MAAS_MODEL: gemma-4-26b-a4b-it-maas
Login-E-Mail im Browser waehlen: niclascramer@gmail.com
ADC OK
configured_project: project-e784413d-cd26-4398-adf
detected_project: project-e784413d-cd26-4398-adf
token_prefix: ya29.a0AQvPy...


## 3. Optional Vertex-Ollama Proxy

Nur ausfuehren, wenn `USE_VERTEX_PROXY=True`. Der Proxy nimmt Ollama-kompatible `/api/chat` Requests vom Runner entgegen und schickt sie zur Google Vertex AI MaaS API. Mit `--force-default-model` werden lokale Namen wie `gemma4:26b` intern auf `VERTEX_MAAS_MODEL` gemappt.


In [7]:
proxy_proc = None
proxy_reused_existing = False

def proxy_is_ready(url: str, timeout: int = 3) -> bool:
    try:
        req = Request(url.rstrip('/') + '/api/tags', method='GET')
        with urlopen(req, timeout=timeout) as response:
            payload = json.loads(response.read().decode('utf-8'))
        print('proxy_ready:', url, payload)
        return True
    except Exception as exc:
        print('proxy_not_ready:', url, exc)
        return False

if not USE_VERTEX_PROXY:
    print('USE_VERTEX_PROXY=False; proxy not started.')
elif proxy_is_ready(PROXY_URL):
    proxy_reused_existing = True
    print('proxy_reused_existing:', PROXY_URL)
else:
    proxy_cmd = [
        sys.executable,
        str(ROOT / 'scripts/vertex_ollama_proxy.py'),
        '--host', '127.0.0.1',
        '--port', '11435',
        '--project-id', PROJECT_ID,
        '--location', LOCATION,
        '--model', VERTEX_MAAS_MODEL,
        '--force-default-model',
    ]

    proxy_env = os.environ.copy()
    proxy_env['PYTHONPATH'] = str(ROOT / 'scripts') + os.pathsep + proxy_env.get('PYTHONPATH', '')

    proxy_proc = subprocess.Popen(
        proxy_cmd,
        cwd=ROOT,
        env=proxy_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    time.sleep(3)
    print('proxy_pid:', proxy_proc.pid)
    if proxy_proc.poll() is not None:
        output = proxy_proc.stdout.read() if proxy_proc.stdout else ''
        print(output)
        if 'Address already in use' in output and proxy_is_ready(PROXY_URL):
            proxy_proc = None
            proxy_reused_existing = True
            print('proxy_reused_existing_after_port_conflict:', PROXY_URL)
        else:
            raise RuntimeError('Proxy exited early')
    elif proxy_is_ready(PROXY_URL):
        print('proxy_started')
    else:
        raise RuntimeError('Proxy process started but /api/tags is not reachable')


proxy_ready: http://127.0.0.1:11435 {'models': [{'name': 'gemma-4-26b-a4b-it-maas'}]}
proxy_reused_existing: http://127.0.0.1:11435


## 4. Optional JSON Smoke Test

Prueft entweder den Vertex-Proxy oder lokales Ollama mit einem kleinen JSON-Call.


In [8]:
smoke_url = PROXY_URL if USE_VERTEX_PROXY else LOCAL_OLLAMA_URL
smoke_model = VERTEX_MAAS_MODEL if USE_VERTEX_PROXY else PLANNER_MODEL
payload = {
    'model': smoke_model,
    'stream': False,
    'format': 'json',
    'messages': [
        {'role': 'system', 'content': 'Return valid JSON only.'},
        {'role': 'user', 'content': 'Return {"ok": true, "model": "gemma"} as JSON.'},
    ],
    'options': {'temperature': 0.0, 'top_p': 0.95, 'num_predict': 128},
}
req = Request(
    smoke_url + '/api/chat',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
started = time.perf_counter()
with urlopen(req, timeout=600) as resp:
    raw = resp.read().decode('utf-8')
elapsed = time.perf_counter() - started
decoded = json.loads(raw)
print('endpoint:', smoke_url)
print('model:', smoke_model)
print('elapsed_seconds:', round(elapsed, 3))
print('prompt_tokens:', decoded.get('prompt_eval_count'))
print('completion_tokens:', decoded.get('eval_count'))
print('content:', decoded.get('message', {}).get('content'))


endpoint: http://127.0.0.1:11435
model: gemma-4-26b-a4b-it-maas
elapsed_seconds: 0.915
prompt_tokens: 39
completion_tokens: 20
content: {
  "ok": true,
  "model": "gemma"
}


## 5. Run The H/k Sweep

Diese Zelle startet den angefragten Sweep mit `v2_planact`. Die Anzahl der Runs ergibt sich aus `len(TASK_IDS) * len(HS) * len(KS)` und wird als `EXPECTED_RUNS` angezeigt. Wenn `USE_VERTEX_PROXY=True`, geht der Lauf ueber Google Vertex AI MaaS; sonst ueber lokales Ollama. MUTATE-Aufgaben werden in den Artefakten besonders markiert, weil hier nicht nur Navigation, sondern ein beobachtbarer State-Change oder eine korrekte Policy-Antwort zaehlt.

`MAX_STEPS_POLICY = "tiered"` nutzt reproduzierbare Budgetklassen nach Capability-Tier: navigation=10, retrieval=20, policy=15, mutation=30. `MAX_PLANNER_CALLS = "auto"` wird danach pro H-Wert und pro Step-Budget aufgeloest: fuer `h>0` gilt `ceil(max_steps(task) / h) + margin`; fuer `h=0` wird nur ein kleiner Runtime-Replan-Puffer genutzt.

Die Run-Zelle streamt stdout/stderr live und liest periodisch die aktuelle `summary.csv`, um Fortschritt, ungefaehre ETA und den letzten abgeschlossenen Run zu zeigen.


In [15]:
import math
import selectors
import pandas as pd

cmd = [
    'uv', 'run', 'python', 'scripts/run_hk_agent_experiment.py',
    '--experiment-name', EXPERIMENT_NAME,
    '--task-ids', *[str(task_id) for task_id in TASK_IDS],
    '--hs', *[str(h) for h in HS],
    '--ks', *[str(k) for k in KS],
    '--run-mode', RUN_MODE,
    '--planner-model', PLANNER_MODEL,
    '--executor-model', EXECUTOR_MODEL,
    '--agent-architecture', AGENT_ARCHITECTURE,
    '--max-steps-policy', MAX_STEPS_POLICY,
    '--max-steps-navigation', str(MAX_STEPS_BY_CAPABILITY_TIER['navigation']),
    '--max-steps-retrieval', str(MAX_STEPS_BY_CAPABILITY_TIER['structured_retrieve']),
    '--max-steps-policy-task', str(MAX_STEPS_BY_CAPABILITY_TIER['policy']),
    '--max-steps-mutation', str(MAX_STEPS_BY_CAPABILITY_TIER['mutation']),
    '--max-planner-calls', str(MAX_PLANNER_CALLS),
    '--planner-call-margin', str(PLANNER_CALL_MARGIN),
    '--max-steps', str(MAX_STEPS),
    '--llm-timeout-seconds', str(LLM_TIMEOUT_SECONDS),
    '--max-consecutive-llm-timeouts', str(MAX_CONSECUTIVE_LLM_TIMEOUTS),
    '--success-policy', SUCCESS_POLICY,
]

runner_env = os.environ.copy()
runner_env['PYTHONPATH'] = str(ROOT / 'scripts') + os.pathsep + runner_env.get('PYTHONPATH', '')
if USE_VERTEX_PROXY:
    cmd.extend(['--ollama-base-url', PROXY_URL])
else:
    cmd.extend(['--ollama-base-url', LOCAL_OLLAMA_URL])
if RESUME_SUMMARY:
    cmd.append('--resume-summary')
if REFRESH_EXISTING_DIAGNOSTICS:
    cmd.append('--refresh-existing-diagnostics')
if REFRESH_EXISTING_ONLY:
    cmd.append('--refresh-existing-only')
if RESET_SITE_BEFORE_MUTATE:
    cmd.append('--reset-site-before-mutate')
    cmd.extend(['--site-reset-timeout-seconds', str(SITE_RESET_TIMEOUT_SECONDS)])

summary_csv = ROOT / 'runs/hk-agent' / EXPERIMENT_NAME / 'summary.csv'
current_matrix_keys = pd.MultiIndex.from_product(
    [[int(task_id) for task_id in TASK_IDS], [int(h) for h in HS], [int(k) for k in KS]],
    names=['task_id', 'h', 'k'],
).to_frame(index=False)
print('architecture:', AGENT_ARCHITECTURE)
print('webarena_prompt_root:', WEBARENA_VERIFIED_PROMPT_ROOT)
print('architecture_prompt_root:', ARCHITECTURE_EXTENSION_PROMPT_ROOT)
print('tasks:', len(TASK_IDS), 'hs:', HS, 'ks:', KS, 'expected_runs:', EXPECTED_RUNS)
print('max_steps_policy:', MAX_STEPS_POLICY, 'fallback_max_steps:', MAX_STEPS)
print('max_steps_by_capability_tier:', MAX_STEPS_BY_CAPABILITY_TIER)
print('max_planner_calls:', MAX_PLANNER_CALLS, 'planner_call_margin:', PLANNER_CALL_MARGIN)
print('llm_timeout_seconds:', LLM_TIMEOUT_SECONDS, 'max_consecutive_llm_timeouts:', MAX_CONSECUTIVE_LLM_TIMEOUTS)
print('reset_site_before_mutate:', RESET_SITE_BEFORE_MUTATE, 'site_reset_timeout_seconds:', SITE_RESET_TIMEOUT_SECONDS)
print('use_contamination_adjusted_success:', USE_CONTAMINATION_ADJUSTED_SUCCESS)
print('success_policy:', SUCCESS_POLICY)
print('refresh_existing_diagnostics:', REFRESH_EXISTING_DIAGNOSTICS)
print('refresh_existing_only:', REFRESH_EXISTING_ONLY)
print('planner_call_budget_by_tier_and_h:', PLANNER_CALL_BUDGET_BY_TIER_AND_H)
print('summary_csv:', summary_csv)
print(' '.join(cmd))

if PRINT_COMMAND_ONLY:
    print('PRINT_COMMAND_ONLY=True; command was printed above and the notebook will not start the sweep.')
else:
    run_started_perf = time.perf_counter()
    run_started_wall = time.time()
    last_progress_print = 0.0
    last_completed_rows = 0

    def progress_snapshot():
        if not summary_csv.exists():
            return None
        # Avoid reading stale results before the current process has written its first summary.
        if summary_csv.stat().st_mtime < run_started_wall:
            return None
        try:
            frame = pd.read_csv(summary_csv)
        except Exception:
            return None
        if frame.empty:
            return None
        frame = frame.drop_duplicates(['task_id', 'h', 'k'], keep='last')
        frame_current = frame.merge(current_matrix_keys, on=['task_id', 'h', 'k'], how='inner')
        completed = len(frame_current)
        elapsed = time.perf_counter() - run_started_perf
        avg_per_run = elapsed / completed if completed else None
        remaining = max(EXPECTED_RUNS - completed, 0)
        eta_s = remaining * avg_per_run if avg_per_run else None
        last = frame_current.iloc[-1] if not frame_current.empty else frame.iloc[-1]
        return {
            'completed': completed,
            'remaining': remaining,
            'elapsed': elapsed,
            'avg_per_run': avg_per_run,
            'eta_s': eta_s,
            'last_task': last.get('task_id'),
            'last_h': last.get('h'),
            'last_k': last.get('k'),
            'last_status': last.get('status'),
            'last_success': last.get('official_success'),
            'last_failure': last.get('failure_category'),
        }

    def format_duration(seconds):
        if seconds is None or not math.isfinite(seconds):
            return 'unknown'
        seconds = int(max(seconds, 0))
        hours, rem = divmod(seconds, 3600)
        minutes, secs = divmod(rem, 60)
        if hours:
            return f'{hours}h {minutes}m {secs}s'
        if minutes:
            return f'{minutes}m {secs}s'
        return f'{secs}s'

    def print_progress(force=False):
        global last_progress_print, last_completed_rows
        now = time.perf_counter()
        snap = progress_snapshot()
        if snap is None:
            if force or now - last_progress_print >= 30:
                print(f'[progress] waiting for current summary.csv write; elapsed={format_duration(now - run_started_perf)}')
                last_progress_print = now
            return
        should_print = force or snap['completed'] != last_completed_rows or now - last_progress_print >= 30
        if not should_print:
            return
        last_completed_rows = snap['completed']
        last_progress_print = now
        pct = 100 * snap['completed'] / EXPECTED_RUNS if EXPECTED_RUNS else 0
        print(
            '[progress] '
            f"{snap['completed']}/{EXPECTED_RUNS} ({pct:.1f}%) done, "
            f"remaining={snap['remaining']}, elapsed={format_duration(snap['elapsed'])}, "
            f"avg/run={format_duration(snap['avg_per_run'])}, eta={format_duration(snap['eta_s'])}, "
            f"last=task {snap['last_task']} h{snap['last_h']} k{snap['last_k']} "
            f"status={snap['last_status']} success={snap['last_success']} failure={snap['last_failure']}"
        )

    proc = subprocess.Popen(
        cmd,
        cwd=ROOT,
        env=runner_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    print('runner_pid:', proc.pid)
    selector = selectors.DefaultSelector()
    if proc.stdout is not None:
        selector.register(proc.stdout, selectors.EVENT_READ)

    output_tail = []
    try:
        while proc.poll() is None:
            events = selector.select(timeout=1.0)
            for key, _mask in events:
                line = key.fileobj.readline()
                if line:
                    print(line, end='')
                    output_tail.append(line)
                    output_tail = output_tail[-400:]
            print_progress()

        if proc.stdout is not None:
            for line in proc.stdout:
                print(line, end='')
                output_tail.append(line)
                output_tail = output_tail[-400:]
    finally:
        try:
            selector.close()
        except Exception:
            pass

    elapsed = time.perf_counter() - run_started_perf
    print_progress(force=True)
    print('returncode:', proc.returncode)
    print('elapsed_seconds:', round(elapsed, 3))
    print('summary_csv:', summary_csv)
    if proc.returncode != 0:
        print('output_tail:', ''.join(output_tail[-120:]))
        raise RuntimeError('H/k sweep failed; see live output/output_tail above.')


architecture: v3
webarena_prompt_root: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/examples/prompts
architecture_prompt_root: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/prompts/v2
tasks: 15 hs: [0, 2, 5, 10] ks: [0, 2, 5, 10] expected_runs: 240
max_steps_policy: tiered fallback_max_steps: 500
max_steps_by_capability_tier: {'navigation': 45, 'visible_retrieve': 45, 'structured_retrieve': 45, 'policy': 45, 'mutation': 150}
max_planner_calls: auto planner_call_margin: 2
llm_timeout_seconds: 600 max_consecutive_llm_timeouts: 0
reset_site_before_mutate: False site_reset_timeout_seconds: 180
use_contamination_adjusted_success: True
success_policy: contamination_adjusted
refresh_existing_diagnostics: True
refresh_existing_only: False
planner_call_budget_by_tier_and_h: {'navigation': {0: 3, 2: 25, 5: 11, 10: 7}, 'visible_retrieve': {0: 3, 2: 25, 5: 11, 10: 7}, 'structured_retrieve': {0: 3, 2: 25, 5: 11, 10: 7}, 'policy':

## 6. Read Summary

This final analysis starts from the existing experiment `summary.csv`. If available, notebook-level success correction via `apply_analysis_success_columns(df)` is applied before any aggregation. The automatic column detection below keeps the analysis reusable across older and newer summaries.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    warnings.warn(f"matplotlib is not available: {exc}")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except Exception as exc:
    px = None
    go = None
    make_subplots = None
    warnings.warn(f"plotly is not available; matplotlib fallbacks will be used where possible: {exc}")

SUMMARY_PATH = ROOT / "runs/hk-agent" / EXPERIMENT_NAME / "summary.csv"
EXPERIMENT_DIR = SUMMARY_PATH.parent
PREFERRED_HK_ORDER = [0, 2, 5, 10]


def detect_column(df, candidates, required=False, label="", warn_missing=True):
    """Return the first available column from candidates, or warn clearly."""
    for col in candidates:
        if col in df.columns:
            return col
    readable = label or "column"
    message = f"No {readable} column found. Tried: {', '.join(candidates)}"
    if required:
        raise KeyError(message)
    if warn_missing:
        warnings.warn(message)
    return None


def minmax_norm(series):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    if values.notna().sum() == 0:
        return pd.Series(0.0, index=series.index)
    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)
    if not np.isfinite(min_value) or not np.isfinite(max_value) or np.isclose(max_value, min_value):
        return pd.Series(0.0, index=series.index)
    return ((values - min_value) / (max_value - min_value)).clip(0, 1).fillna(0.0)


def robust_minmax_norm(series, upper_quantile=0.95):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    upper = values.quantile(upper_quantile) if values.notna().any() else np.nan
    if np.isfinite(upper):
        values = values.clip(upper=upper)
    return minmax_norm(values)


def _as_numeric_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(float)
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().any():
        return numeric.fillna(0).clip(lower=0, upper=1).astype(float)
    truthy = {"true", "1", "yes", "y", "success", "succeeded", "pass", "passed"}
    return series.fillna(False).astype(str).str.strip().str.lower().isin(truthy).astype(float)


def _series_or_default(df, col, default=0.0):
    if col and col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(default).astype(float)
    return pd.Series(default, index=df.index, dtype="float64")


def load_summary(path=SUMMARY_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(f"Summary not found: {path}")
    summary = pd.read_csv(path).copy()
    success_func = globals().get("apply_analysis_success_columns")
    if callable(success_func):
        summary = success_func(summary)
    else:
        warnings.warn("apply_analysis_success_columns(df) is not defined; using success columns already present in summary.csv.")
    return summary


def prepare_eval_df(summary):
    df = summary.copy()

    success_col = detect_column(
        df,
        ["official_success_bool", "official_success", "success", "contamination_adjusted_success"],
        required=True,
        label="success",
    )
    h_col = detect_column(df, ["h", "H", "planning_horizon"], required=True, label="H")
    k_col = detect_column(df, ["k", "K", "validation_interval", "replanning_interval"], required=True, label="k")
    task_col = detect_column(df, ["task_id"], required=False, label="task id")
    site_col = detect_column(df, ["site"], required=False, label="site")

    token_col = detect_column(df, ["total_tokens", "tokens", "total_token_count"], required=False, label="token", warn_missing=False)
    if token_col:
        token_values = _series_or_default(df, token_col)
        token_label = token_col
    elif {"prompt_tokens", "completion_tokens"}.issubset(df.columns):
        token_values = _series_or_default(df, "prompt_tokens") + _series_or_default(df, "completion_tokens")
        token_label = "prompt_tokens + completion_tokens"
    else:
        warnings.warn("No token column found; token normalization is set to 0 and token efficiency has no effect.")
        token_values = pd.Series(0.0, index=df.index, dtype="float64")
        token_label = None

    runtime_col = detect_column(df, ["total_runtime_s", "runtime_seconds"], required=False, label="runtime seconds", warn_missing=False)
    runtime_ms_col = None
    if runtime_col:
        runtime_values = _series_or_default(df, runtime_col)
        runtime_label = runtime_col
    elif "total_runtime_ms" in df.columns:
        runtime_values = _series_or_default(df, "total_runtime_ms") / 1000.0
        runtime_label = "total_runtime_ms / 1000"
        runtime_ms_col = "total_runtime_ms"
    elif "duration_s" in df.columns:
        runtime_values = _series_or_default(df, "duration_s")
        runtime_label = "duration_s"
    elif "duration_ms" in df.columns:
        runtime_values = _series_or_default(df, "duration_ms") / 1000.0
        runtime_label = "duration_ms / 1000"
        runtime_ms_col = "duration_ms"
    else:
        warnings.warn("No runtime column found; runtime normalization is set to 0 and time efficiency has no effect.")
        runtime_values = pd.Series(0.0, index=df.index, dtype="float64")
        runtime_label = None

    eval_df = df.copy()
    eval_df["h"] = pd.to_numeric(df[h_col], errors="coerce")
    eval_df["k"] = pd.to_numeric(df[k_col], errors="coerce")
    eval_df["success"] = _as_numeric_bool(df[success_col])
    eval_df["tokens"] = token_values
    eval_df["runtime_s"] = runtime_values
    eval_df["site"] = df[site_col].fillna("unknown").astype(str) if site_col else "unknown"
    if task_col:
        eval_df["task_id"] = df[task_col]
    eval_df["candidate_id"] = "H=" + eval_df["h"].map("{:.0f}".format) + ", k=" + eval_df["k"].map("{:.0f}".format)

    before = len(eval_df)
    eval_df = eval_df.dropna(subset=["h", "k"]).copy()
    if len(eval_df) != before:
        warnings.warn(f"Dropped {before - len(eval_df)} rows without numeric H/k values.")
    eval_df["h"] = eval_df["h"].astype(int)
    eval_df["k"] = eval_df["k"].astype(int)

    duplicate_keys = [col for col in ["task_id", "h", "k"] if col in eval_df.columns]
    if len(duplicate_keys) == 3:
        duplicated = eval_df.duplicated(duplicate_keys, keep=False)
        if duplicated.any():
            warnings.warn(f"Found {int(duplicated.sum())} duplicate task/H/k rows; keeping the last row per task/H/k.")
            eval_df = eval_df.drop_duplicates(duplicate_keys, keep="last")

    eval_df.attrs["success_col"] = success_col
    eval_df.attrs["token_col"] = token_label
    eval_df.attrs["runtime_col"] = runtime_label
    eval_df.attrs["h_col"] = h_col
    eval_df.attrs["k_col"] = k_col
    eval_df.attrs["task_col"] = task_col
    eval_df.attrs["site_col"] = site_col
    return eval_df


def ordered_values(values, preferred=PREFERRED_HK_ORDER):
    unique = sorted(pd.Series(values).dropna().astype(int).unique().tolist())
    if set(unique).issubset(set(preferred)):
        return [x for x in preferred if x in unique]
    return unique


summary = load_summary()
eval_df = prepare_eval_df(summary)

summary_overview = pd.DataFrame([
    {"metric": "summary_path", "value": str(SUMMARY_PATH.relative_to(ROOT))},
    {"metric": "rows", "value": len(eval_df)},
    {"metric": "unique_tasks", "value": eval_df["task_id"].nunique() if "task_id" in eval_df.columns else np.nan},
    {"metric": "h_values", "value": ordered_values(eval_df["h"])},
    {"metric": "k_values", "value": ordered_values(eval_df["k"])},
    {"metric": "sites", "value": sorted(eval_df["site"].dropna().unique().tolist())},
    {"metric": "global_success_rate", "value": eval_df["success"].mean()},
    {"metric": "avg_tokens", "value": eval_df["tokens"].mean()},
    {"metric": "avg_runtime_s", "value": eval_df["runtime_s"].mean()},
    {"metric": "success_column", "value": eval_df.attrs.get("success_col")},
    {"metric": "token_column", "value": eval_df.attrs.get("token_col")},
    {"metric": "runtime_column", "value": eval_df.attrs.get("runtime_col")},
])

display(summary_overview)

display_cols = [
    "task_id", "site", "h", "k", "success", "tokens", "runtime_s", "total_steps",
    "failure_category", "output_dir",
]
display(eval_df[[col for col in display_cols if col in eval_df.columns]].head(20))


## 7. Final Utility Definition

The final analysis uses the success-dominant utility

$$U(H,k) = S(H,k) \cdot [\alpha + \beta \cdot (1 - T_{norm}(H,k)) + \gamma \cdot (1 - \tau_{norm}(H,k))]$$

with $\alpha + \beta + \gamma = 1$.

Success is the primary metric. Token cost and runtime are secondary efficiency criteria. H and k are not penalized directly; the utility only evaluates their observed empirical effect on success, token use, and runtime. Failed configurations should not receive a positive score just because they are cheap or fast, so success multiplies the efficiency term from the outside. `T_norm` and `tau_norm` are clipped to `[0, 1]`; the final utility is not clipped.

The weight profiles are not learned model parameters. They are evaluation preferences. The main analysis uses a success-dominant profile, and alternative profiles test whether the optimal H/k combination remains robust when token cost or runtime receives more weight.

`k=0` is treated as an experimental special case. Depending on the runner setup, it can mean a baseline/no-validation/no-explicit-validation-interval condition and should be interpreted methodologically with that caveat.

In [ ]:
PERSONAS = {
    "main_accuracy_first": {"alpha": 0.90, "beta": 0.05, "gamma": 0.05},
    "cost_sensitive": {"alpha": 0.80, "beta": 0.15, "gamma": 0.05},
    "time_sensitive": {"alpha": 0.80, "beta": 0.05, "gamma": 0.15},
    "stress_80": {"alpha": 0.75, "beta": 0.125, "gamma": 0.125},
}

for name, weights in PERSONAS.items():
    total = sum(weights.values())
    if not np.isclose(total, 1.0):
        raise ValueError(f"Persona {name} weights must sum to 1, got {total}")


## 8. Aggregate H/k Results

In [ ]:
def aggregate_hk(eval_df, group_cols=None):
    if group_cols is None:
        group_cols = ["h", "k"]
    group_cols = list(group_cols)
    missing = [col for col in group_cols if col not in eval_df.columns]
    if missing:
        raise KeyError(f"Cannot aggregate; missing columns: {missing}")

    task_agg = ("task_id", "nunique") if "task_id" in eval_df.columns else ("success", "size")
    agg_spec = {
        "runs": ("success", "size"),
        "tasks": task_agg,
        "success_rate": ("success", "mean"),
        "avg_tokens": ("tokens", "mean"),
        "median_tokens": ("tokens", "median"),
        "avg_runtime_s": ("runtime_s", "mean"),
        "median_runtime_s": ("runtime_s", "median"),
    }
    if "total_steps" in eval_df.columns:
        agg_spec["avg_steps"] = ("total_steps", "mean")

    hk_df = eval_df.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
    hk_df["T_norm"] = minmax_norm(hk_df["avg_tokens"])
    hk_df["runtime_norm"] = minmax_norm(hk_df["avg_runtime_s"])
    hk_df["candidate_id"] = "H=" + hk_df["h"].astype(int).astype(str) + ", k=" + hk_df["k"].astype(int).astype(str)

    sort_cols = [col for col in ["site", "h", "k"] if col in hk_df.columns]
    return hk_df.sort_values(sort_cols).reset_index(drop=True)


def compute_utility(row, alpha, beta, gamma):
    return row["success_rate"] * (alpha + beta * (1 - row["T_norm"]) + gamma * (1 - row["runtime_norm"]))


def compute_utility_columns(hk_df, personas):
    result = hk_df.copy()
    for persona, weights in personas.items():
        utility_col = f"U_{persona}"
        rank_col = f"rank_{persona}"
        result[utility_col] = result.apply(lambda row: compute_utility(row, **weights), axis=1)
        result[rank_col] = result[utility_col].rank(method="min", ascending=False).astype(int)
    return result


hk_global = aggregate_hk(eval_df)
hk_global = compute_utility_columns(hk_global, PERSONAS)

utility_cols = [f"U_{name}" for name in PERSONAS]
rank_cols = [f"rank_{name}" for name in PERSONAS]
base_cols = ["h", "k", "candidate_id", "runs", "tasks", "success_rate", "avg_tokens", "median_tokens", "avg_runtime_s", "median_runtime_s", "T_norm", "runtime_norm"]
if "avg_steps" in hk_global.columns:
    base_cols.insert(base_cols.index("avg_tokens"), "avg_steps")

display(
    hk_global[base_cols + utility_cols + rank_cols]
    .sort_values("U_main_accuracy_first", ascending=False)
    .style.format({
        "success_rate": "{:.3f}",
        "avg_tokens": "{:.0f}",
        "median_tokens": "{:.0f}",
        "avg_runtime_s": "{:.1f}",
        "median_runtime_s": "{:.1f}",
        "T_norm": "{:.3f}",
        "runtime_norm": "{:.3f}",
        **{col: "{:.4f}" for col in utility_cols},
    })
)

hk_by_site = None
if "site" in eval_df.columns:
    hk_by_site = aggregate_hk(eval_df, group_cols=["site", "h", "k"])
    hk_by_site = compute_utility_columns(hk_by_site, PERSONAS)


## 9. Utility Personas / Weight Profiles

In [ ]:
def get_persona_winners(hk_df, personas):
    rows = []
    main_best = None
    if "U_main_accuracy_first" in hk_df.columns:
        main_row = hk_df.sort_values("U_main_accuracy_first", ascending=False).iloc[0]
        main_best = (int(main_row["h"]), int(main_row["k"]))

    for persona, weights in personas.items():
        utility_col = f"U_{persona}"
        rank_col = f"rank_{persona}"
        if utility_col not in hk_df.columns:
            raise KeyError(f"Missing utility column: {utility_col}")
        best = hk_df.sort_values([utility_col, "success_rate"], ascending=[False, False]).iloc[0]
        best_pair = (int(best["h"]), int(best["k"]))
        rows.append({
            "persona": persona,
            "alpha": weights["alpha"],
            "beta": weights["beta"],
            "gamma": weights["gamma"],
            "best_h": best_pair[0],
            "best_k": best_pair[1],
            "success_rate": best["success_rate"],
            "avg_tokens": best["avg_tokens"],
            "avg_runtime_s": best["avg_runtime_s"],
            "utility": best[utility_col],
            "rank": int(best[rank_col]) if rank_col in hk_df.columns else 1,
            "same_as_main_best": best_pair == main_best,
        })
    return pd.DataFrame(rows)


persona_winners = get_persona_winners(hk_global, PERSONAS)

display(hk_global[base_cols + utility_cols + rank_cols].sort_values("U_main_accuracy_first", ascending=False))
display(persona_winners.style.format({
    "alpha": "{:.3f}",
    "beta": "{:.3f}",
    "gamma": "{:.3f}",
    "success_rate": "{:.3f}",
    "avg_tokens": "{:.0f}",
    "avg_runtime_s": "{:.1f}",
    "utility": "{:.4f}",
}))


## 10. Top H/k Candidates

In [ ]:
def get_top_candidates(hk_df, utility_col="U_main_accuracy_first", n=5):
    if utility_col not in hk_df.columns:
        raise KeyError(f"Missing utility column: {utility_col}")
    return (
        hk_df.sort_values([utility_col, "success_rate"], ascending=[False, False])
        .head(n)
        .copy()
        .reset_index(drop=True)
    )


top3_candidates = get_top_candidates(hk_global, n=3)
top5_candidates = get_top_candidates(hk_global, n=5)

top_candidate_cols = [
    "candidate_id", "h", "k", "runs", "tasks", "success_rate", "avg_tokens", "avg_runtime_s",
    "T_norm", "runtime_norm", *utility_cols, *rank_cols,
]

display(Markdown("**Top 3 H/k candidates by main utility**"))
display(top3_candidates[top_candidate_cols].style.format({
    "success_rate": "{:.3f}",
    "avg_tokens": "{:.0f}",
    "avg_runtime_s": "{:.1f}",
    "T_norm": "{:.3f}",
    "runtime_norm": "{:.3f}",
    **{col: "{:.4f}" for col in utility_cols},
}))

display(Markdown("**Top 5 H/k candidates by main utility**"))
display(top5_candidates[top_candidate_cols].style.format({
    "success_rate": "{:.3f}",
    "avg_tokens": "{:.0f}",
    "avg_runtime_s": "{:.1f}",
    "T_norm": "{:.3f}",
    "runtime_norm": "{:.3f}",
    **{col: "{:.4f}" for col in utility_cols},
}))


## 11. 3D Spike Map: Utility over H/k

In [ ]:
def plot_3d_spike_map(hk_df, utility_col, title, top_candidates=None, output_path=None):
    plot_df = hk_df.copy()
    top_ids = set(top_candidates["candidate_id"].tolist()) if top_candidates is not None and "candidate_id" in top_candidates else set()

    if go is not None:
        fig = go.Figure()
        for _, row in plot_df.iterrows():
            color = "crimson" if row["candidate_id"] in top_ids else "steelblue"
            width = 8 if row["candidate_id"] in top_ids else 5
            fig.add_trace(go.Scatter3d(
                x=[row["h"], row["h"]],
                y=[row["k"], row["k"]],
                z=[0, row[utility_col]],
                mode="lines",
                line=dict(color=color, width=width),
                showlegend=False,
                hovertext=f"{row['candidate_id']}<br>{utility_col}: {row[utility_col]:.4f}<br>success: {row['success_rate']:.3f}",
                hoverinfo="text",
            ))
        fig.add_trace(go.Scatter3d(
            x=plot_df["h"], y=plot_df["k"], z=plot_df[utility_col],
            mode="markers+text",
            marker=dict(size=6, color=plot_df[utility_col], colorscale="Viridis", showscale=True, colorbar=dict(title="Utility")),
            text=[cid if cid in top_ids else "" for cid in plot_df["candidate_id"]],
            textposition="top center",
            name="H/k",
        ))
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title="Planning horizon H",
                yaxis_title="Validation interval k",
                zaxis_title="Utility",
                xaxis=dict(tickmode="array", tickvals=ordered_values(plot_df["h"])),
                yaxis=dict(tickmode="array", tickvals=ordered_values(plot_df["k"])),
            ),
            margin=dict(l=0, r=0, t=60, b=0),
        )
        if output_path is not None:
            fig.write_html(str(output_path))
        fig.show()
        return fig

    if plt is None:
        warnings.warn("Neither plotly nor matplotlib is available; skipping 3D spike map.")
        return None

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    colors = ["crimson" if cid in top_ids else "steelblue" for cid in plot_df["candidate_id"]]
    ax.bar3d(plot_df["h"], plot_df["k"], np.zeros(len(plot_df)), 0.5, 0.5, plot_df[utility_col], color=colors, shade=True)
    ax.set_title(title)
    ax.set_xlabel("Planning horizon H")
    ax.set_ylabel("Validation interval k")
    ax.set_zlabel("Utility")
    display(fig)
    return fig


spike_outputs = {}
for persona in PERSONAS:
    utility_col = f"U_{persona}"
    output_path = EXPERIMENT_DIR / f"final_analysis_3d_spike_{persona}.html"
    spike_outputs[persona] = output_path
    fig = plot_3d_spike_map(
        hk_global,
        utility_col=utility_col,
        title=f"3D spike map: {persona} utility over H/k",
        top_candidates=top5_candidates,
        output_path=output_path if go is not None else None,
    )
    if persona == "main_accuracy_first":
        main_spike_fig = fig


## 12. Weight Sensitivity of Top Candidates

In [ ]:
def compute_weight_sensitivity(top_candidates, personas_or_slices=None):
    slices = [
        {"slice": "alpha_0_90", "alpha": 0.90, "beta_min": 0.00, "beta_max": 0.10, "step": 0.01},
        {"slice": "alpha_0_80", "alpha": 0.80, "beta_min": 0.00, "beta_max": 0.20, "step": 0.01},
        {"slice": "alpha_0_75", "alpha": 0.75, "beta_min": 0.00, "beta_max": 0.25, "step": 0.01},
    ] if personas_or_slices is None else personas_or_slices

    rows = []
    for spec in slices:
        beta_values = np.round(np.arange(spec["beta_min"], spec["beta_max"] + spec["step"] / 2, spec["step"]), 2)
        for beta in beta_values:
            alpha = spec["alpha"]
            gamma = round(1 - alpha - beta, 10)
            if gamma < -1e-9:
                continue
            for _, candidate in top_candidates.iterrows():
                rows.append({
                    "slice": spec["slice"],
                    "alpha": alpha,
                    "beta": beta,
                    "gamma": gamma,
                    "candidate_id": candidate["candidate_id"],
                    "h": int(candidate["h"]),
                    "k": int(candidate["k"]),
                    "utility": compute_utility(candidate, alpha=alpha, beta=beta, gamma=gamma),
                })
    return pd.DataFrame(rows)


def plot_top_candidate_sensitivity(sensitivity_df):
    if sensitivity_df.empty:
        warnings.warn("Sensitivity table is empty; skipping plots.")
        return None
    if px is not None:
        fig = px.line(
            sensitivity_df,
            x="beta",
            y="utility",
            color="candidate_id",
            facet_col="slice",
            markers=True,
            title="Weight sensitivity of top H/k candidates (gamma = 1 - alpha - beta)",
            labels={"beta": "Token weight beta", "utility": "Utility"},
        )
        fig.update_yaxes(matches=None)
        fig.show()
        return fig
    if plt is None:
        return None
    figures = []
    for slice_name, slice_df in sensitivity_df.groupby("slice"):
        fig, ax = plt.subplots(figsize=(7, 4))
        for candidate_id, candidate_df in slice_df.groupby("candidate_id"):
            ax.plot(candidate_df["beta"], candidate_df["utility"], marker="o", label=candidate_id)
        alpha_value = slice_df["alpha"].iloc[0]
        ax.set_title(f"{slice_name}: alpha={alpha_value:.2f}, gamma=1-alpha-beta")
        ax.set_xlabel("Token weight beta")
        ax.set_ylabel("Utility")
        ax.legend()
        display(fig)
        figures.append(fig)
    return figures


weight_sensitivity_top_candidates = compute_weight_sensitivity(top3_candidates)
display(weight_sensitivity_top_candidates.head(20))
sensitivity_fig = plot_top_candidate_sensitivity(weight_sensitivity_top_candidates)


## 13. Full Weight-Space Winner Map

In [ ]:
def compute_weight_space_winner_map(hk_df, alpha_min=0.75, alpha_max=1.0, step=0.01):
    rows = []
    alpha_values = np.round(np.arange(alpha_min, alpha_max + step / 2, step), 2)
    for alpha in alpha_values:
        max_beta = 1 - alpha
        beta_values = np.round(np.arange(0, max_beta + step / 2, step), 2)
        for beta in beta_values:
            gamma = round(1 - alpha - beta, 10)
            if gamma < -1e-9:
                continue
            utilities = hk_df.apply(lambda row: compute_utility(row, alpha=alpha, beta=beta, gamma=gamma), axis=1)
            best_idx = utilities.idxmax()
            best = hk_df.loc[best_idx]
            rows.append({
                "alpha": alpha,
                "beta": beta,
                "gamma": gamma,
                "best_h": int(best["h"]),
                "best_k": int(best["k"]),
                "best_candidate_id": best["candidate_id"],
                "best_utility": utilities.loc[best_idx],
            })
    return pd.DataFrame(rows)


def plot_weight_space_winner_map(winner_map, fixed_profiles=None):
    if winner_map.empty:
        warnings.warn("Winner map is empty; skipping plot.")
        return None
    fixed_profiles = fixed_profiles or {}
    if px is not None:
        fig = px.scatter(
            winner_map,
            x="beta",
            y="gamma",
            color="best_candidate_id",
            hover_data=["alpha", "best_h", "best_k", "best_utility"],
            title="Full weight-space winner map (alpha = 1 - beta - gamma)",
            labels={"beta": "Token weight beta", "gamma": "Runtime weight gamma"},
        )
        for name, weights in fixed_profiles.items():
            fig.add_trace(go.Scatter(
                x=[weights["beta"]], y=[weights["gamma"]], mode="markers+text",
                text=[name], textposition="top center",
                marker=dict(size=12, color="black", symbol="x"),
                name=name,
            ))
        fig.show()
        return fig
    if plt is None:
        return None
    fig, ax = plt.subplots(figsize=(8, 6))
    candidate_codes = winner_map["best_candidate_id"].astype("category")
    scatter = ax.scatter(winner_map["beta"], winner_map["gamma"], c=candidate_codes.cat.codes, cmap="tab20", s=35)
    for name, weights in fixed_profiles.items():
        ax.scatter(weights["beta"], weights["gamma"], marker="x", color="black", s=80)
        ax.text(weights["beta"], weights["gamma"], name)
    ax.set_title("Full weight-space winner map")
    ax.set_xlabel("Token weight beta")
    ax.set_ylabel("Runtime weight gamma")
    display(fig)
    return fig


def plot_winner_counts(winner_map):
    counts = (
        winner_map["best_candidate_id"]
        .value_counts(normalize=True)
        .mul(100)
        .rename_axis("candidate_id")
        .reset_index(name="winner_share_pct")
    )
    counts["winner_count"] = winner_map["best_candidate_id"].value_counts().reindex(counts["candidate_id"]).values
    if px is not None:
        fig = px.bar(
            counts,
            x="candidate_id",
            y="winner_share_pct",
            text="winner_share_pct",
            title="Winner frequency across evaluated weight space",
            labels={"winner_share_pct": "Winner share (%)", "candidate_id": "H/k candidate"},
        )
        fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig.show()
    elif plt is not None:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(counts["candidate_id"], counts["winner_share_pct"])
        ax.set_title("Winner frequency across evaluated weight space")
        ax.set_ylabel("Winner share (%)")
        ax.tick_params(axis="x", rotation=45)
        display(fig)
    return counts


weight_space_winner_map = compute_weight_space_winner_map(hk_global)
weight_space_fig = plot_weight_space_winner_map(weight_space_winner_map, PERSONAS)

winner_counts = plot_winner_counts(weight_space_winner_map)
display(weight_space_winner_map.head(20))
display(winner_counts)

if go is not None and weight_space_fig is not None:
    weight_space_fig.write_html(str(EXPERIMENT_DIR / "final_analysis_weight_space_winner_map.html"))


## 14. Site-Level Comparison

In [ ]:
def get_site_winners(hk_by_site, personas):
    if hk_by_site is None or hk_by_site.empty:
        return pd.DataFrame()
    rows = []
    for site, site_df in hk_by_site.groupby("site", dropna=False):
        best = site_df.sort_values("U_main_accuracy_first", ascending=False).iloc[0]
        rows.append({
            "site": site,
            "best_h": int(best["h"]),
            "best_k": int(best["k"]),
            "runs": int(best["runs"]),
            "tasks": int(best["tasks"]),
            "success_rate": best["success_rate"],
            "avg_tokens": best["avg_tokens"],
            "avg_runtime_s": best["avg_runtime_s"],
            "U_main_accuracy_first": best["U_main_accuracy_first"],
        })
    return pd.DataFrame(rows).sort_values("site").reset_index(drop=True)


site_winners = get_site_winners(hk_by_site, PERSONAS)
if site_winners.empty:
    display(Markdown("No `site` column found; site-level comparison skipped."))
else:
    display(site_winners.style.format({
        "success_rate": "{:.3f}",
        "avg_tokens": "{:.0f}",
        "avg_runtime_s": "{:.1f}",
        "U_main_accuracy_first": "{:.4f}",
    }))
    display(hk_by_site.sort_values(["site", "U_main_accuracy_first"], ascending=[True, False]).head(40))


## 15. Export Final Analysis Tables

In [ ]:
exports = {
    "final_analysis_hk_global.csv": hk_global,
    "final_analysis_persona_winners.csv": persona_winners,
    "final_analysis_top_candidates.csv": top5_candidates,
    "final_analysis_weight_sensitivity_top_candidates.csv": weight_sensitivity_top_candidates,
    "final_analysis_weight_space_winner_map.csv": weight_space_winner_map,
    "final_analysis_winner_counts.csv": winner_counts,
}
if hk_by_site is not None:
    exports["final_analysis_hk_by_site.csv"] = hk_by_site
if not site_winners.empty:
    exports["final_analysis_site_winners.csv"] = site_winners

exported_files = []
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
for filename, table in exports.items():
    output_path = EXPERIMENT_DIR / filename
    table.to_csv(output_path, index=False)
    exported_files.append(str(output_path.relative_to(ROOT)))

display(pd.DataFrame({"exported_file": exported_files}))
if go is not None:
    display(pd.DataFrame({"html_plot": [str(path.relative_to(ROOT)) for path in spike_outputs.values()]}))


## 16. Automated Markdown Interpretation

In [ ]:
def automated_interpretation(hk_global, persona_winners, top3_candidates, winner_counts):
    lines = []
    main_best = hk_global.sort_values("U_main_accuracy_first", ascending=False).iloc[0]
    main_id = main_best["candidate_id"]
    stable_profiles = int(persona_winners["same_as_main_best"].sum())
    top3_text = ", ".join(top3_candidates["candidate_id"].tolist())

    lines.append(f"The best H/k combination by main utility is **{main_id}**.")
    lines.append(
        f"It reaches a success rate of **{main_best['success_rate']:.3f}**, "
        f"with **{main_best['avg_tokens']:.0f}** average tokens and "
        f"**{main_best['avg_runtime_s']:.1f}s** average runtime."
    )
    lines.append(f"This combination remains optimal in **{stable_profiles} of {len(PERSONAS)}** fixed weight profiles.")
    lines.append(f"The Top-3 candidates by main utility are: **{top3_text}**.")

    if not winner_counts.empty:
        most_common = winner_counts.iloc[0]
        lines.append(
            f"Across the full evaluated weight space, **{most_common['candidate_id']}** wins in "
            f"**{most_common['winner_share_pct']:.1f}%** of weighting points."
        )

    cost_best = persona_winners.loc[persona_winners["persona"].eq("cost_sensitive")].iloc[0]
    time_best = persona_winners.loc[persona_winners["persona"].eq("time_sensitive")].iloc[0]
    lines.append(f"With stronger token-cost weighting, the optimum is **H={int(cost_best['best_h'])}, k={int(cost_best['best_k'])}**.")
    lines.append(f"With stronger runtime weighting, the optimum is **H={int(time_best['best_h'])}, k={int(time_best['best_k'])}**.")
    return "\n\n".join(lines)


display(Markdown(automated_interpretation(hk_global, persona_winners, top3_candidates, winner_counts)))


## 17. Stop Proxy

Only relevant if `USE_VERTEX_PROXY=True` and this notebook kernel started the proxy process.

In [ ]:
if globals().get("proxy_proc") is None:
    if globals().get("proxy_reused_existing"):
        print("Existing proxy was reused; leaving it running.")
    else:
        print("No proxy process to stop.")
else:
    proxy_proc.terminate()
    try:
        proxy_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proxy_proc.kill()
    print("proxy_stopped")
